In [2]:
import ee
ee.Initialize(project="sentinel-487715")

# Ghana boundary
ghana = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
    .filter(ee.Filter.eq("country_na", "Ghana")) \
    .geometry()

# Roads asset
roads = ee.FeatureCollection("projects/sentinel-487715/assets/ghana_roads")

# Quarter ranges
quarters = {
    "Q1": ("01-01","03-31"),
    "Q2": ("04-01","06-30"),
    "Q3": ("07-01","09-30"),
    "Q4": ("10-01","12-31"),
}

def make_img(start, end):
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(ghana)
          .filterDate(start, end)
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
          .select(["B2","B3","B4","B8","B11","B12"])
          .median())

    ndvi = s2.normalizedDifference(["B8","B4"]).rename("NDVI")
    ndmi = s2.normalizedDifference(["B8","B11"]).rename("NDMI")
    ndbi = s2.normalizedDifference(["B11","B8"]).rename("NDBI")
    ndwi = s2.normalizedDifference(["B3","B8"]).rename("NDWI")
    bsi = (s2.select("B11").add(s2.select("B4"))
           .subtract(s2.select("B8").add(s2.select("B2")))
           .divide(s2.select("B11").add(s2.select("B4"))
                   .add(s2.select("B8")).add(s2.select("B2")))
           .rename("BSI"))

    return s2.addBands([ndvi, ndmi, ndbi, ndwi, bsi])

# Launch exports
tasks = []
for year in range(2020, 2024):
    for q, (start_mmdd, end_mmdd) in quarters.items():
        start = f"{year}-{start_mmdd}"
        end = f"{year}-{end_mmdd}"

        img = make_img(start, end)
        reducer = ee.Reducer.mean().forEachBand(img)

        stats_fc = img.reduceRegions(
            collection=roads,
            reducer=reducer,
            scale=20,
            tileScale=16
        ).map(lambda f: f.set({"year": year, "quarter": q}))

        desc = f"ghana_roads_{year}_{q}"
        task = ee.batch.Export.table.toDrive(
            collection=stats_fc,
            description=desc,
            fileFormat="CSV",
            folder="GEE_Exports"
        )
        task.start()
        tasks.append(task)

print(f"Started {len(tasks)} export tasks.")


Started 16 export tasks.


In [1]:
# Export with cloud threshold of 40% for Q3 (2020–2023)

import ee
ee.Initialize(project="sentinel-487715")

# Ghana boundary
ghana = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
    .filter(ee.Filter.eq("country_na", "Ghana")) \
    .geometry()

# Roads asset
roads = ee.FeatureCollection("projects/sentinel-487715/assets/ghana_roads")

def make_img_q3(start, end, cloud_thresh=40):
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(ghana)
          .filterDate(start, end)
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_thresh))
          .select(["B2","B3","B4","B8","B11","B12"])
          .median())

    ndvi = s2.normalizedDifference(["B8","B4"]).rename("NDVI")
    ndmi = s2.normalizedDifference(["B8","B11"]).rename("NDMI")
    ndbi = s2.normalizedDifference(["B11","B8"]).rename("NDBI")
    ndwi = s2.normalizedDifference(["B3","B8"]).rename("NDWI")
    bsi = (s2.select("B11").add(s2.select("B4"))
           .subtract(s2.select("B8").add(s2.select("B2")))
           .divide(s2.select("B11").add(s2.select("B4"))
                   .add(s2.select("B8")).add(s2.select("B2")))
           .rename("BSI"))

    return s2.addBands([ndvi, ndmi, ndbi, ndwi, bsi])

tasks = []
for year in range(2020, 2024):
    start = f"{year}-07-01"
    end   = f"{year}-09-30"

    img = make_img_q3(start, end, cloud_thresh=40)
    reducer = ee.Reducer.mean().forEachBand(img)

    stats_fc = img.reduceRegions(
        collection=roads,
        reducer=reducer,
        scale=20,
        tileScale=16
    ).map(lambda f: f.set({"year": year, "quarter": "Q3", "cloud_thresh": 40}))

    desc = f"ghana_roads_{year}_Q3_cloud40"
    task = ee.batch.Export.table.toDrive(
        collection=stats_fc,
        description=desc,
        fileNamePrefix=desc,
        folder="GEE_Exports",
        fileFormat="CSV"
    )
    task.start()
    tasks.append(task)
    print("Started:", desc, "| task id:", task.id)

print(f"Started {len(tasks)} Q3 export tasks (2020–2023, cloud < 40%).")


Started: ghana_roads_2020_Q3_cloud40 | task id: BDE6G6FRR3QMH5ZKG7IBX4PK
Started: ghana_roads_2021_Q3_cloud40 | task id: SJOVWGBJSIFVQXUWCFCDK7T2
Started: ghana_roads_2022_Q3_cloud40 | task id: 4QBVTTRHS5J3FELNF4QNIOFO
Started: ghana_roads_2023_Q3_cloud40 | task id: BEAGX7HZORMG2XZAGXZQTY5D
Started 4 Q3 export tasks (2020–2023, cloud < 40%).


In [ ]:
# export for Nigeria with all attributes and 2020 quarterly stats

import ee
ee.Initialize(project="sentinel-487715")

# -----------------------------
# Assets (update paths if needed)
# -----------------------------
RC = ee.FeatureCollection("projects/sentinel-487715/assets/nigeria/RoadCondition")
RN = ee.FeatureCollection("projects/sentinel-487715/assets/nigeria/RoadNetwork")
RP = ee.FeatureCollection("projects/sentinel-487715/assets/nigeria/RoadPavement")
PA = ee.FeatureCollection("projects/sentinel-487715/assets/nigeria/ProblemAreas")
DR = ee.FeatureCollection("projects/sentinel-487715/assets/nigeria/RoadSideDrains")
CU = ee.FeatureCollection("projects/sentinel-487715/assets/nigeria/Culverts")
BR = ee.FeatureCollection("projects/sentinel-487715/assets/nigeria/Bridges")

EXPORT_FOLDER = "GEE_Exports"
CLOUD_MAX = 40
SCALE_M = 20
TILE_SCALE = 16

# -----------------------------
# Normalize ROADCODE
# -----------------------------
def with_code(fc):
    def _f(f):
        c = ee.String(ee.Algorithms.If(f.get("ROADCODE"), f.get("ROADCODE"), ""))
        c = c.trim().replace("\\s+", "")
        return f.set("ROADCODE_N", c)
    return fc.map(_f)

def with_patype(fc):
    def _f(f):
        p = ee.String(ee.Algorithms.If(f.get("PATYPE"), f.get("PATYPE"), ""))
        return f.set("PATYPE_N", p.trim().upper())
    return fc.map(_f)

rc = with_code(RC)
rn = with_code(RN)
rp = with_code(RP)
pa = with_patype(with_code(PA))
dr = with_code(DR)
cu = with_code(CU)
br = with_code(BR)

# -----------------------------
# Geometry backbone = RoadCondition
# Add attrs/counts from all other layers
# -----------------------------
def add_attrs(f):
    code = ee.String(f.get("ROADCODE_N"))

    rn1 = ee.Feature(rn.filter(ee.Filter.eq("ROADCODE_N", code)).first())
    rp1 = ee.Feature(rp.filter(ee.Filter.eq("ROADCODE_N", code)).first())
    pa_c = pa.filter(ee.Filter.eq("ROADCODE_N", code))

    return f.set({
        "ROADCLASS": rn1.get("ROADCLASS"),
        "TOTALLEN": rn1.get("TOTALLEN"),
        "SHPLEN": rn1.get("SHPLEN"),
        "PAVETYPE": rp1.get("PAVETYPE"),
        "PAVEWIDTH": rp1.get("PAVEWIDTH"),
        "FORWIDTH": rp1.get("FORWIDTH"),
        "waterlogging_n": pa_c.filter(ee.Filter.eq("PATYPE_N", "WATERLOGGING")).size(),
        "erosion_n": pa_c.filter(ee.Filter.eq("PATYPE_N", "EROSION")).size(),
        "landslide_n": pa_c.filter(ee.Filter.eq("PATYPE_N", "LANDSLIDE")).size(),
        "drains_n": dr.filter(ee.Filter.eq("ROADCODE_N", code)).size(),
        "culverts_n": cu.filter(ee.Filter.eq("ROADCODE_N", code)).size(),
        "bridges_n": br.filter(ee.Filter.eq("ROADCODE_N", code)).size()
    })

roads_full = rc.map(add_attrs)
aoi = roads_full.geometry().bounds()

# -----------------------------
# Sentinel image builder
# -----------------------------
def make_img(start_date, end_date):
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(aoi)
          .filterDate(start_date, end_date)
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_MAX))
          .select(["B2", "B3", "B4", "B8", "B11", "B12"])
          .median())

    ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = s2.normalizedDifference(["B8", "B11"]).rename("NDMI")
    ndbi = s2.normalizedDifference(["B11", "B8"]).rename("NDBI")
    ndwi = s2.normalizedDifference(["B3", "B8"]).rename("NDWI")
    bsi = (s2.select("B11").add(s2.select("B4"))
           .subtract(s2.select("B8").add(s2.select("B2")))
           .divide(s2.select("B11").add(s2.select("B4"))
                   .add(s2.select("B8")).add(s2.select("B2")))
           .rename("BSI"))

    return s2.addBands([ndvi, ndmi, ndbi, ndwi, bsi])

# -----------------------------
# 2020 quarterly exports only
# -----------------------------
quarters = {
    "Q1": ("2020-01-01", "2020-03-31"),
    "Q2": ("2020-04-01", "2020-06-30"),
    "Q3": ("2020-07-01", "2020-09-30"),
    "Q4": ("2020-10-01", "2020-12-31"),
}

tasks = []
for q, (start, end) in quarters.items():
    img = make_img(start, end)
    reducer = ee.Reducer.mean().forEachBand(img)

    stats_fc = img.reduceRegions(
        collection=roads_full,
        reducer=reducer,
        scale=SCALE_M,
        tileScale=TILE_SCALE
    ).map(lambda f: f.set({"year": 2020, "quarter": q}))

    desc = f"nigeria_roads_s2_2020_{q}"
    task = ee.batch.Export.table.toDrive(
        collection=stats_fc,
        description=desc,
        folder=EXPORT_FOLDER,
        fileNamePrefix=desc,
        fileFormat="CSV"
    )
    task.start()
    tasks.append(task)
    print("Started:", desc, task.id)

print("Total tasks started:", len(tasks))
